# 06 · Inside the CMFM block

No training here. We open up the Cross-Modal Fusion Mamba block and check, with tensors in hand, the properties the method claims: how big each part is, what a fresh block computes, that a zero-reliability sensor really cannot write into the scan's state, how the tokens are ordered, and that the gated-convolution control is matched to CFFM-Net in compute.

| | |
|---|---|
| **Kaggle settings** | Accelerator **GPU T4 x2**, Internet **on** |
| **Inputs to attach** | `cffm-net-pilot` |
| **Expected runtime** | about 2 minutes |
| **Produces** | nothing; this notebook is for understanding and checking |

In [ ]:
# --------------------------------------------------------------------------------------------
# Setup: the same cell opens every notebook. It finds the code, installs it, and checks the GPU.
#
# On Kaggle the uploaded cffm-net-pilot.zip appears, already unzipped, somewhere under
# /kaggle/input. Inputs are read-only, so we copy the project to /kaggle/working and install
# it from there ("pip install -e" makes `import cffm` work in this kernel, in DataLoader
# workers, and in the extra processes Ultralytics starts when it trains on two GPUs).
# Locally, notebooks live in cffm-net-pilot/notebooks/<phase>/; the project is the first folder up
# that contains src/cffm.
# --------------------------------------------------------------------------------------------
import glob, shutil, subprocess, sys
from pathlib import Path

ON_KAGGLE = Path("/kaggle/working").exists()
if ON_KAGGLE:
    PROJECT = Path("/kaggle/working/cffm-net-pilot")
    if not PROJECT.exists():
        # Up to five levels deep: /kaggle/input/<slug>/, /kaggle/input/<slug>/cffm-net-pilot/,
        # /kaggle/input/datasets/<owner>/<slug>/..., whichever layout this Kaggle version uses.
        hits = [Path(p).parent for k in range(1, 6) for p in glob.glob("/kaggle/input" + "/*" * k + "/pyproject.toml")
                if (Path(p).parent / "src" / "cffm").is_dir()]
        # Saved outputs of earlier notebooks also hold a copy of the code, next to their runs/ or data/.
        # Prefer the uploaded dataset, so a stale copy can never win.
        hits.sort(key=lambda h: any((h.parent / d).exists() for d in ("runs", "data")))
        zips = [Path(p) for k in range(1, 5) for p in glob.glob("/kaggle/input" + "/*" * k + ".zip")
                if "cffm" in Path(p).name]
        if hits:
            shutil.copytree(hits[0], PROJECT, copy_function=shutil.copyfile)
        elif zips:                       # the zip was attached but not unpacked by Kaggle
            shutil.unpack_archive(str(zips[0]), "/kaggle/working")   # it holds one folder, cffm-net-pilot/
        assert PROJECT.exists(), "Add the 'cffm-net-pilot' dataset (the uploaded zip) as an input to this notebook."
        for p in [PROJECT, *PROJECT.rglob("*")]:     # inputs are read-only; our copy must be writable
            p.chmod(p.stat().st_mode | 0o200)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "ultralytics==8.4.171",
                    "faster-coco-eval>=1.6.7", "cloudpickle", "pytest"], check=True)   # cloudpickle: two-GPU launcher
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", str(PROJECT)], check=True)
else:
    PROJECT = next(p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (p / "src" / "cffm").is_dir())
sys.path.insert(0, str(PROJECT / "src"))

import cffm
from cffm import env as cenv, pipeline
print("cffm", cffm.__version__, "imported from", Path(cffm.__file__).parent)   # must be inside PROJECT
env = cenv.setup()                       # prints platform, GPUs and paths; moves into the project folder
plan = pipeline.load_plan(env.project)   # configs/pilot.yaml: every run and its settings

## 0. The method in three pictures

**CFFM-Net.** There are two YOLO26-n backbones, one per sensor, with separate weights; both start from COCO.
They are fused at four levels, from P2 (stride 4) to P5 (stride 32). P2 uses a reliability-weighted sum, and
P3 to P5 use CMFM. The fused maps feed YOLO26's stride-4 PAN neck and its NMS-free heads.

In [ ]:
from IPython.display import Image, display
for name in ('fig_architecture',):
    f = env.project / "docs" / "figures" / f"{name}.png"
    if f.exists():
        display(Image(filename=str(f), width=700))
    else:
        print("missing", f)

**CMFM.** One reliability head scores each sensor at each location, and that score has two jobs: it gates
the scan and it weights the residual. Thermal features are first shifted by a learned offset field. The two
streams are then interleaved token by token and scanned in four directions. The output projection starts at
zero, so a fresh block is exactly the reliability-weighted average.

In [ ]:
from IPython.display import Image, display
for name in ('fig_cmfm',):
    f = env.project / "docs" / "figures" / f"{name}.png"
    if f.exists():
        display(Image(filename=str(f), width=640))
    else:
        print("missing", f)

**The gated recurrence.** Each token's step $\Delta_k$ is multiplied by the reliability $r_k$ of its sensor.
As $r_k \to 0$, $\bar A_k \to 1$ and $\bar B_k \to 0$, so the token neither writes into the state nor
erases it. Section 3 below checks this numerically.

In [ ]:
from IPython.display import Image, display
for name in ('fig_gated_scan',):
    f = env.project / "docs" / "figures" / f"{name}.png"
    if f.exists():
        display(Image(filename=str(f), width=640))
    else:
        print("missing", f)

## 1. Where the parameters go

One CMFM block at the stride-8 level of YOLO26-n (128 channels).

In [ ]:
import torch, pandas as pd
from cffm.blocks import CMFM, GatedCrossScan
torch.manual_seed(0)
blk = CMFM(128, impl="torch").eval()
rows = [{"part": n, "params": sum(p.numel() for p in m.parameters())} for n, m in blk.named_children()]
df = pd.DataFrame(rows); df.loc[len(df)] = {"part": "total", "params": df["params"].sum()}; df

## 2. A fresh block is a reliability-weighted average

The output projection starts at zero and both reliabilities start at sigmoid(2) = 0.88, so before training CMFM returns exactly (F_v + F_t) / 2. Training can only *add* to that safe starting point.

In [ ]:
fv, ft = torch.randn(2, 128, 40, 40), torch.randn(2, 128, 40, 40)
z = blk(fv, ft)
print("max |z - (fv + ft)/2| =", (z - (fv + ft) / 2).abs().max().item())

## 3. The property everything rests on

Set the thermal reliability to zero. Every thermal token then gets a step of zero, so it can neither write
into the scan's state nor erase it, and the outputs at visible positions must not change however much we
perturb the thermal features. With reliability one, the same perturbation must change them.

In [ ]:
scan = GatedCrossScan(16, directions=4, impl="torch").eval()
xv, xt = torch.randn(1, 16, 12, 12), torch.randn(1, 16, 12, 12)
ones, zeros = torch.ones(1, 1, 12, 12), torch.zeros(1, 1, 12, 12)
noise = 5 * torch.randn_like(xt)
d_off = (scan(xv, xt, ones, zeros)[0] - scan(xv, xt + noise, ones, zeros)[0]).abs().max().item()
d_on = (scan(xv, xt, ones, ones)[0] - scan(xv, xt + noise, ones, ones)[0]).abs().max().item()
print(f"r_thermal = 0: visible outputs move by {d_off:.2e}   (should be ~0)")
print(f"r_thermal = 1: visible outputs move by {d_on:.2e}   (should be clearly > 0)")

## 4. Token order

The first scan direction on a 2 x 3 map, written as (row, column, modality). Visible (v) and thermal (t) alternate at every location: fusion happens inside the state.

In [ ]:
X = torch.zeros(1, 1, 2, 3, 2)
for r in range(2):
    for c in range(3):
        X[0, 0, r, c, 0], X[0, 0, r, c, 1] = 10 * r + c, 100 + 10 * r + c
seq = GatedCrossScan(1, directions=4)._orders(X)[0, 0, 0].int().tolist()
print(" ".join(f"{'t' if s >= 100 else 'v'}({(s % 100) // 10},{s % 10})" for s in seq))

## 5. Compute of every model, and the matched control

GFLOPs at 640 x 640 include the element-wise work inside the scans (profilers do not count it; see `cffm.train.gflops`). The gated-convolution control (C4) should sit within a few percent of CFFM-Net.

In [ ]:
from cffm.model import DualStreamDetectionModel
from cffm.train import gflops, scan_gflops
rows = []
for cfg in sorted((env.project / "configs" / "models").glob("*.yaml")):
    m = DualStreamDetectionModel(str(cfg), nc=1, verbose=False)
    rows.append({"model": cfg.stem, "params (M)": round(sum(p.numel() for p in m.parameters()) / 1e6, 3),
                 "GFLOPs": round(gflops(m, 640), 2), "of which scans": round(scan_gflops(m, 640), 3)})
df = pd.DataFrame(rows).set_index("model"); print(df)
ratio = df.loc["cffm-net-n-gconv", "GFLOPs"] / df.loc["cffm-net-n", "GFLOPs"]
print(f"control / CFFM-Net GFLOPs = {ratio:.3f}"); assert abs(ratio - 1) < 0.05